# Chapter 5: Data Preparation


## Notebook 5.2: Case Study Application - VIXY Tail Hedge Strategy

**Financial Trading with Python, 2nd Edition**

---



### Notebook Overview

This notebook applies the data preparation principles from Notebook 5.1 to our VIXY tail hedge case study. The key lesson: simple strategies need simple preparation.

The VIXY strategy uses four instruments:
- SPY: S&P 500 ETF (the risky asset)
- BIL: Treasury bills ETF (the safe asset)
- VIXY: VIX short-term futures ETF (the tail hedge)
- ^VIX: The VIX index (for signal generation only, not traded)

We will load the data saved in Chapter 3, verify its quality, perform the minimal transformations needed, and save the prepared data for use in later chapters.



### Learning Objectives

By the end of this notebook, you will:
- See that clean data from a reliable source needs little preparation
- Understand what transformations the VIXY strategy actually requires
- Have prepared data ready for signal generation and backtesting

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.4f}'.format)

## 1. Loading the Case Study Data

In Chapter 3, we downloaded price data for our four instruments and saved it as a parquet file. Let us load it and see what we have.

Note: Make sure to drag and drop the parquet file into the Colab instance!

In [2]:
# Load the case study data from Chapter 4
df_prices = pd.read_parquet('case_study_prices.parquet')

print("=== Data Overview ===")
print(f"Shape: {df_prices.shape}")
print(f"Date range: {df_prices.index.min().date()} to {df_prices.index.max().date()}")
print()
print("Columns:")
print(df_prices.columns.tolist())
print()
print("First 5 rows:")
print(df_prices.head())

=== Data Overview ===
Shape: (3771, 4)
Date range: 2011-01-04 to 2025-12-31

Columns:
['BIL', 'SPY', 'VIXY', '^VIX']

First 5 rows:
               BIL     SPY        VIXY    ^VIX
Date                                          
2011-01-04 74.9744 97.1394 633840.0000 17.3800
2011-01-05 74.9744 97.6442 620400.0000 17.0200
2011-01-06 74.9744 97.4530 623040.0000 17.4000
2011-01-07 74.9580 97.2618 624320.0000 17.1400
2011-01-10 74.9744 97.1394 623040.0000 17.5400


We have 3,771 rows of daily data from January 2011 through December 2025. The data starts in 2011 because that is when VIXY began trading.

Notice the VIXY prices in early 2011 are extremely high (over 600,000). This is not an error. VIXY has undergone multiple reverse splits over the years to keep its price tradeable as the ETF decayed. The historical prices are adjusted for these splits, so a share purchased in 2011 would have been worth that much in today's share-equivalent terms.

## 2. Inspecting Data Quality

Before making any changes, let us verify what we have. We will check for missing values, examine basic statistics, and look for obvious problems.

In [3]:
# Check for missing values
print("=== Missing Values ===")
print(df_prices.isna().sum())
print()

# Basic statistics
print("=== Summary Statistics ===")
print(df_prices.describe())

=== Missing Values ===
BIL     0
SPY     0
VIXY    0
^VIX    0
dtype: int64

=== Summary Statistics ===
            BIL       SPY        VIXY      ^VIX
count 3771.0000 3771.0000   3771.0000 3771.0000
mean    77.9499  281.0401  67143.1828   18.1367
std      4.3376  153.5977 159368.5151    6.8805
min     74.6670   85.3193     25.4700    9.1400
25%     74.8795  162.1801    347.2000   13.5950
50%     75.8444  241.0381   2410.3999   16.3500
75%     78.4670  393.6828  32160.0000   20.5750
max     91.3800  690.3800 975600.0000   82.6900


The data is clean:

- No missing values in any column. The alignment was already handled in Chapter 3 when we used `dropna()` to keep only dates where all four instruments had data.
- BIL (Treasury bills) is very stable, ranging from 74.67 to 91.38. This is expected for a cash-like instrument.
- SPY ranges from 85.32 to 690.38, reflecting market growth over 15 years.
- VIXY shows extreme variation from 25.47 to 975,600 due to reverse splits and the ETF's natural decay.
- VIX ranges from 9.14 to 82.69. The maximum of 82.69 occurred during the COVID crash in March 2020.

There are no negative values, no obvious errors, and no gaps. This is what you should expect from a reputable data source with proper alignment.

### 2.1 Checking Index Integrity

Let us verify that the date index is properly formatted.

In [4]:
# Check index integrity
print("=== Index Checks ===")
print(f"Index type: {type(df_prices.index)}")
print(f"Index name: {df_prices.index.name}")
print(f"Is monotonic increasing: {df_prices.index.is_monotonic_increasing}")
print(f"Has duplicates: {df_prices.index.has_duplicates}")
print()

# Check for gaps larger than expected (more than 5 business days)
date_diff = df_prices.index.to_series().diff().dt.days
large_gaps = date_diff[date_diff > 5]
print(f"Gaps larger than 5 days: {len(large_gaps)}")
if len(large_gaps) > 0:
    print(large_gaps.head(10))

=== Index Checks ===
Index type: <class 'pandas.core.indexes.datetimes.DatetimeIndex'>
Index name: Date
Is monotonic increasing: True
Has duplicates: False

Gaps larger than 5 days: 0


The index passes all checks:

- Proper DatetimeIndex type
- Sorted in ascending order (monotonic increasing)
- No duplicate dates
- No unexpected gaps longer than 5 days

The data is ready. We did not need to fix anything because the data was properly prepared when we saved it in Chapter 3.

## 3. What Does the VIXY Strategy Need?

Now we ask the key question: what preparation does this specific strategy require?

The VIXY tail hedge strategy generates signals based on:
1. The 90-day moving average of VIX
2. The 10-day realized volatility of SPY (annualized)

When SPY's realized volatility exceeds the VIX moving average, the strategy allocates to VIXY as a tail hedge. Let us calculate these features.

In [5]:
# Create a copy for our prepared data
df_prepared = df_prices.copy()

# Calculate SPY daily returns
df_prepared['spy_return'] = df_prepared['SPY'].pct_change()

# Calculate 10-day realized volatility of SPY (annualized)
df_prepared['spy_realized_vol'] = df_prepared['spy_return'].rolling(10).std() * np.sqrt(252)

# Convert to percentage for comparison with VIX
df_prepared['spy_realized_vol_pct'] = df_prepared['spy_realized_vol'] * 100

# Calculate 90-day moving average of VIX
df_prepared['vix_ma_90'] = df_prepared['^VIX'].rolling(90).mean()

print("=== Prepared Data ===")
print(df_prepared[['SPY', 'spy_return', 'spy_realized_vol_pct', '^VIX', 'vix_ma_90']].iloc[88:98])

=== Prepared Data ===
                SPY  spy_return  spy_realized_vol_pct    ^VIX  vix_ma_90
Date                                                                    
2011-05-11 103.2930     -0.0105               10.1425 16.9500        NaN
2011-05-12 103.7847      0.0048               10.3424 16.0300    17.9312
2011-05-13 102.9856     -0.0077               10.7249 17.0700    17.9278
2011-05-16 102.3326     -0.0063               10.9672 18.2400    17.9413
2011-05-17 102.3172     -0.0002               10.9830 17.5500    17.9430
2011-05-18 103.2315      0.0089               11.8442 16.2300    17.9329
2011-05-19 103.4774      0.0024               10.8335 15.5200    17.9104
2011-05-20 102.6553     -0.0079               11.4442 17.4300    17.9164
2011-05-23 101.4644     -0.0116               12.4148 18.2700    17.9390
2011-05-24 101.3799     -0.0008               11.0068 17.8200    17.9549


The first valid row for all calculations is 2011-05-12 (row 89). This is because:
- spy_return needs 1 prior day
- spy_realized_vol needs 10 days of returns
- vix_ma_90 needs 90 days of VIX data

The 90-day requirement is the binding constraint. We lose the first 90 rows to warm-up, leaving 3,681 rows for analysis.

Looking at these sample rows, SPY realized volatility is around 10-12% (annualized) while the VIX 90-day moving average is around 17.9%. Since realized volatility is below the VIX moving average, the strategy would not be allocating to VIXY during this period.

### 3.1 Verifying No Lookahead Bias

Both of our calculations use `.rolling()`, which only looks backward. Let us verify this explicitly.

In [6]:
# Verify no lookahead bias by checking that calculations only use past data
# Calculate on partial data (first 500 rows)
df_partial = df_prices.iloc[:500].copy()
df_partial['spy_return'] = df_partial['SPY'].pct_change()
df_partial['spy_realized_vol_pct'] = df_partial['spy_return'].rolling(10).std() * np.sqrt(252) * 100
df_partial['vix_ma_90'] = df_partial['^VIX'].rolling(90).mean()

# Compare row 250 between partial and full calculations
row_idx = 250
print("Lookahead Bias Test at row 250:")
print()
print(f"spy_realized_vol_pct (full data):    {df_prepared['spy_realized_vol_pct'].iloc[row_idx]:.4f}")
print(f"spy_realized_vol_pct (partial data): {df_partial['spy_realized_vol_pct'].iloc[row_idx]:.4f}")
print(f"Match: {np.isclose(df_prepared['spy_realized_vol_pct'].iloc[row_idx], df_partial['spy_realized_vol_pct'].iloc[row_idx])}")
print()
print(f"vix_ma_90 (full data):    {df_prepared['vix_ma_90'].iloc[row_idx]:.4f}")
print(f"vix_ma_90 (partial data): {df_partial['vix_ma_90'].iloc[row_idx]:.4f}")
print(f"Match: {np.isclose(df_prepared['vix_ma_90'].iloc[row_idx], df_partial['vix_ma_90'].iloc[row_idx])}")

Lookahead Bias Test at row 250:

spy_realized_vol_pct (full data):    19.6813
spy_realized_vol_pct (partial data): 19.6813
Match: True

vix_ma_90 (full data):    31.7929
vix_ma_90 (partial data): 31.7929
Match: True


Both calculations produce identical results whether we use the full dataset or only the first 500 rows. This confirms that row 250's values depend only on data from rows 0-250, not on any future data.

The strategy is lookahead-free by design. Using `.rolling()` ensures each calculation only looks backward.

## 4. Handling Missing Values from Rolling Calculations

The rolling calculations created missing values in the first 90 rows. We need to decide how to handle these.

In [7]:
# Check missing values after transformations
print("=== Missing Values After Transformations ===")
print(df_prepared.isna().sum())
print()

# First valid row for all columns
first_valid = df_prepared.dropna().index[0]
print(f"First row with all valid data: {first_valid.date()}")
print(f"Rows lost to warm-up period: {df_prepared.index.get_loc(first_valid)}")

=== Missing Values After Transformations ===
BIL                      0
SPY                      0
VIXY                     0
^VIX                     0
spy_return               1
spy_realized_vol        10
spy_realized_vol_pct    10
vix_ma_90               89
dtype: int64

First row with all valid data: 2011-05-12
Rows lost to warm-up period: 89


The missing value counts follow logically from our calculations:

- spy_return: 1 missing (first row has no prior price)
- spy_realized_vol: 10 missing (need 10 returns for rolling standard deviation)
- vix_ma_90: 89 missing (need 90 days for rolling mean)

The 90-day VIX moving average is the binding constraint. We lose 89 rows to the warm-up period, leaving us with 3,682 complete rows starting May 12, 2011.

For this strategy, we simply drop the warm-up rows. They cannot generate valid signals anyway.

In [8]:
# Drop rows with missing values
df_prepared = df_prepared.dropna()

print(f"Final dataset shape: {df_prepared.shape}")
print(f"Date range: {df_prepared.index.min().date()} to {df_prepared.index.max().date()}")
print()
print("Missing values after dropna:")
print(df_prepared.isna().sum())

Final dataset shape: (3682, 8)
Date range: 2011-05-12 to 2025-12-31

Missing values after dropna:
BIL                     0
SPY                     0
VIXY                    0
^VIX                    0
spy_return              0
spy_realized_vol        0
spy_realized_vol_pct    0
vix_ma_90               0
dtype: int64


## 5. Validating the Prepared Data

Before saving, let us run our validation checks.

In [9]:
def validate_dataframe(df):
    """Validate the prepared VIXY strategy data."""
    print("Running validation checks...")

    # Check 1: Index is DatetimeIndex
    assert isinstance(df.index, pd.DatetimeIndex), "Index must be DatetimeIndex"
    print("  ✓ Index is DatetimeIndex")

    # Check 2: Index is sorted
    assert df.index.is_monotonic_increasing, "Index must be sorted"
    print("  ✓ Index is sorted")

    # Check 3: No duplicate dates
    assert not df.index.has_duplicates, "Index has duplicate dates"
    print("  ✓ No duplicate dates")

    # Check 4: No missing values
    assert df.isna().sum().sum() == 0, "Missing values present"
    print("  ✓ No missing values")

    # Check 5: Prices are positive
    for col in ['BIL', 'SPY', 'VIXY']:
        assert (df[col] > 0).all(), f"Non-positive values in {col}"
    print("  ✓ All prices positive")

    # Check 6: VIX is in reasonable range (0-100)
    assert (df['^VIX'] > 0).all() and (df['^VIX'] < 150).all(), "VIX out of range"
    print("  ✓ VIX in reasonable range")

    # Check 7: Volatility is positive
    assert (df['spy_realized_vol'] > 0).all(), "Negative volatility"
    print("  ✓ Realized volatility positive")

    print("All validation checks passed.")
    return True

validate_dataframe(df_prepared)

Running validation checks...
  ✓ Index is DatetimeIndex
  ✓ Index is sorted
  ✓ No duplicate dates
  ✓ No missing values
  ✓ All prices positive
  ✓ VIX in reasonable range
  ✓ Realized volatility positive
All validation checks passed.


True

All checks pass. The data is clean, properly formatted, and ready for the next chapter.

## 6. Saving the Prepared Data

We save the prepared data for use in later chapters.

In [10]:
# Save the prepared data
df_prepared.to_parquet('case_study_prepared.parquet')

print("Data saved to case_study_prepared.parquet")
print()
print("Columns saved:")
for col in df_prepared.columns:
    print(f"  - {col}")

Data saved to case_study_prepared.parquet

Columns saved:
  - BIL
  - SPY
  - VIXY
  - ^VIX
  - spy_return
  - spy_realized_vol
  - spy_realized_vol_pct
  - vix_ma_90


## 7. Summary

### What We Did

1. Loaded the aligned price data from Chapter 3
2. Verified it was already clean (no missing values, no errors)
3. Calculated the two features needed for the strategy:
   - 10-day realized volatility of SPY (annualized, in percentage terms)
   - 90-day moving average of VIX
4. Confirmed no lookahead bias in our calculations
5. Dropped the warm-up period rows
6. Validated and saved the prepared data

### What We Did Not Need

- Filling missing values (there were none after alignment)
- Handling outliers (no data errors, and extreme VIX values are real events we want to capture)
- Resampling (strategy operates on daily data)
- Encoding categorical variables (no categorical features)
- Interaction terms (simple rules-based strategy)
- Winsorization (not doing cross-sectional analysis)

### The Lesson

Simple strategies need simple preparation. The VIXY tail hedge is a rules-based strategy with two rolling calculations. We spent more time verifying the data was clean than actually transforming it.

The extensive techniques in Notebook 5.1 exist for when you need them. Complex machine learning models with many features require careful encoding, interaction terms, and attention to lookahead bias in feature engineering. A simple moving average strategy does not.

Know what your strategy needs before applying every technique in the book.